# Stage 1 — Take inventory

**The question:** what do we actually have, and does any of it surprise us?

10,616 rows in `train.csv` and 10,616 files on disk. We do not yet know whether
they all open, how each file is built inside, or whether the two hospitals
scanned at the same magnification. All of it is cheap to find out now, and
every later stage leans on some of it.

Output: `data/derived/slide_inventory.parquet`, one row per slide.

In [1]:
"""Imports.

os        builds file paths and checks whether a file exists on disk.
cv2       (OpenCV) converts the four-channel RGBA image that OpenSlide returns
          into a single-channel grayscale image.
openslide opens the whole-slide .tiff files and reads their pyramid levels,
          dimensions and metadata.
numpy     turns images into arrays and computes the brightness histogram.
pandas    loads train.csv, holds the inventory table, and saves it as parquet.
tqdm      draws a progress bar for the long loop over every slide.
"""

import os

import cv2
import openslide
import numpy as np
import pandas as pd
import tqdm

In [2]:
"""Paths.

Every file location is set here, so no path is written out twice anywhere
else in the notebook. The same notebook runs both locally and on Kaggle.

1. ON_KAGGLE is True when the Kaggle input folder exists, which only happens
   inside a Kaggle notebook.
2. DATA_DIR is where the competition data lives. Kaggle mounts it under
   /kaggle/input; locally it sits in ../data.
3. DERIVED_DIR is where this project writes its own files. On Kaggle it cannot
   sit under DATA_DIR, because /kaggle/input is read-only, so there it goes to
   /kaggle/working instead.
4. CSV_PATH is the labels table, one row per slide.
5. IMAGE_DIR holds the slide images, one {slide_id}.tiff per slide.
6. MASK_DIR holds the label masks, one {slide_id}_mask.tiff for each slide
   that has a mask.
7. INVENTORY_PATH is the file this notebook produces.
"""

ON_KAGGLE = os.path.exists("/kaggle/input")

if ON_KAGGLE:
    DATA_DIR = "/kaggle/input/prostate-cancer-grade-assessment/"
    DERIVED_DIR = "/kaggle/working/derived/"
else:
    DATA_DIR = "../data/"
    DERIVED_DIR = os.path.join(DATA_DIR, "derived/")

CSV_PATH = os.path.join(DATA_DIR, "train.csv")
IMAGE_DIR = os.path.join(DATA_DIR, "train_images")
MASK_DIR = os.path.join(DATA_DIR, "train_label_masks")
INVENTORY_PATH = os.path.join(DERIVED_DIR, "slide_inventory.parquet")

In [3]:
"""Load the training labels.

train.csv has one row per slide and four columns:
- image_id: the slide's name, which is also its file name without ".tiff"
- data_provider: the hospital, "radboud" or "karolinska"
- isup_grade: the grade we are trying to predict, from 0 to 5
- gleason_score: the pathologist's Gleason score, from which the ISUP grade is
  derived

df.head() shows the first five rows, to confirm the file loaded as expected.
"""

df = pd.read_csv(CSV_PATH)
df.head()

,image_id,data_provider,isup_grade,gleason_score
0,0005f7aaab2800f6170c399693a96917,karolinska,0,0+0
1,000920ad0b612851f8e01bcc880d9b3d,karolinska,0,0+0
2,0018ae58b01bdadc8e347995b69f99aa,radboud,4,4+4
3,001c62abd11fa4b57bf7a6c603a11bb9,karolinska,4,4+4
4,001d865e65ef5d2579c190a0e0350d8f,karolinska,0,0+0


In [4]:
"""Confirm the table has the expected size.

The competition describes 10,616 training slides with 4 columns, so this should
print (10616, 4). Anything else means the wrong file was loaded or it is
incomplete.
"""

print(df.shape)

(10616, 4)


## Build the inventory

One row per slide: the labels from `train.csv`, whether the image and mask
files exist, what OpenSlide reports about the file, and a rough "how blank is
this slide" number from the smallest pyramid level.

Roughly eight minutes. Each slide is opened once and closed again — a loop this
long that leaks handles dies partway through with an error that does not name
the cause.

In [5]:
"""Columns of the inventory table, in order.

One entry for each piece of information recorded about a slide. The loop in
the next cell refers to these by position (keys[0] is slide_id, keys[1] is
gleason_score, and so on), so the order of this list matters: do not reorder
it without updating the loop.

Each column is described in detail in the next cell.
"""

keys = [
    "slide_id",
    "gleason_score",
    "isup_grade",
    "provider",
    "slide_exists",
    "mask_exists",
    "feature_file_exists",
    "feature_tile_count",
    "tiff_pym_lvls",
    "tiff_lvl0_dim",
    "tiff_lvl_ds",
    "tiff_mpx",
    "percentage_empty",
]

In [6]:
"""Build the inventory table: one row per slide.

How the loop works
------------------
- rows is the number of slides in train.csv.
- manifest_df is created up front with one empty row per slide and one column
  per entry in keys. The loop fills it in, one row at a time.
- For each slide, new_row starts as a dictionary with every column set to
  None. Because values are filled in by column name, a slide that fails
  halfway still produces a complete row, with None wherever a value could not
  be read, instead of breaking the table.
- At the end of each pass, the finished dictionary is written into row i.

The 13 columns
--------------
From train.csv, with no file access needed:

1. slide_id: taken from the image_id column.
2. gleason_score: taken from the gleason_score column. The two hospitals write
   "no cancer" differently: Radboud writes "negative" and Karolinska writes
   "0+0". Radboud's "negative" is rewritten as "0+0" so that both hospitals
   use the same value.
3. isup_grade: taken from the isup_grade column.
4. provider: taken from the data_provider column.

From the file system:

5. slide_exists: True if {slide_id}.tiff exists in IMAGE_DIR, checked with
   os.path.exists().
6. mask_exists: True if {slide_id}_mask.tiff exists in MASK_DIR, i.e. whether
   this slide has a label mask.

Reserved for later stages, always None in this notebook:

7. feature_file_exists: whether the slide's extracted feature file exists.
8. feature_tile_count: how many tiles that feature file holds.

From the slide file itself. These need the slide to be opened, so they sit
inside a try block: if a file is missing or unreadable, the error is printed,
these columns stay None, and the loop moves on to the next slide instead of
stopping.

9. tiff_pym_lvls: how many pyramid levels the file holds, i.e. how many copies
   of the same image it stores at different sizes.
10. tiff_lvl0_dim: the size of level 0, the full-resolution image, as
    (width, height). Width comes first.
11. tiff_lvl_ds: each level's downsample factor, i.e. how many times smaller
    than level 0 it is. Level 0 is always 1.
12. tiff_mpx: microns per pixel, as (x, y): how much real tissue one pixel
    covers, left to right and top to bottom. OpenSlide stores these as text,
    so float() converts them to numbers.
13. percentage_empty: a rough measure of how much of the slide is blank glass,
    computed in five steps:
    - read the smallest pyramid level in full; it is the only level small
      enough to read whole quickly;
    - convert it to grayscale, so each pixel becomes one brightness value from
      0 (black) to 255 (white);
    - split the 0-255 range into 8 equal bins of 32 values each, and count how
      many pixels fall into each bin;
    - divide each count by the total, so the bins become fractions adding up
      to 1;
    - take the last bin, brightness 224 to 255, which is near-white, i.e.
      empty glass, and multiply it by 100 to get a percentage.
    This is a rough ranking for later stages, not a tissue detector.

Closing the slide
-----------------
slide is reset to None at the start of each pass, and the finally block closes
it only if this pass actually opened one. finally runs whether or not an error
occurred, so no slide is ever left open. Leaving slides open would eventually
use up the operating system's limit on open files partway through the loop,
with an error message that does not name the cause.
"""

rows = df.shape[0]

manifest_df = pd.DataFrame(index=range(rows), columns=keys)

for i in tqdm.tqdm(range(rows)):
    new_row = {a: None for a in keys}
    new_row[keys[0]] = df.loc[i]["image_id"]
    gleason_score = df.loc[i]["gleason_score"]
    if gleason_score == "negative":
        new_row[keys[1]] = "0+0"
    else:
        new_row[keys[1]] = gleason_score

    new_row[keys[2]] = df.loc[i]["isup_grade"]
    new_row[keys[3]] = df.loc[i]["data_provider"]
    new_row[keys[4]] = os.path.exists(os.path.join(IMAGE_DIR, f"{new_row[keys[0]]}.tiff"))
    new_row[keys[5]] = os.path.exists(os.path.join(MASK_DIR, f"{new_row[keys[0]]}_mask.tiff"))
    new_row[keys[6]] = None
    new_row[keys[7]] = None

    slide = None
    try:
        slide = openslide.OpenSlide(os.path.join(IMAGE_DIR, f"{new_row[keys[0]]}.tiff"))
        new_row[keys[8]] = slide.level_count
        new_row[keys[9]] = slide.level_dimensions[0]
        new_row[keys[10]] = slide.level_downsamples
        new_row[keys[11]] = (
            float(slide.properties.get("openslide.mpp-x")),
            float(slide.properties.get("openslide.mpp-y")),
        )
        image = slide.read_region((0, 0), slide.level_count - 1, slide.level_dimensions[slide.level_count - 1])
        image = cv2.cvtColor(np.array(image), cv2.COLOR_RGBA2GRAY)
        histogram, _ = np.histogram(image, bins=8, range=(0, 256))
        histogram = histogram / histogram.sum()
        histogram = histogram.tolist()
        percentage_empty = histogram[-1] * 100
        new_row[keys[12]] = percentage_empty
    except Exception as e:
        print(f"Slide: {new_row[keys[0]]}  ;\n Exception: {e}")
    finally:
        if slide is not None:
            slide.close()

    manifest_df.loc[i] = new_row

100%|██████████| 10616/10616 [07:33<00:00, 23.40it/s]


In [7]:
"""Show the first five rows of the inventory, to check by eye that every
column was filled in with sensible values."""

manifest_df.head()

,slide_id,gleason_score,isup_grade,provider,slide_exists,mask_exists,feature_file_exists,feature_tile_count,tiff_pym_lvls,tiff_lvl0_dim,tiff_lvl_ds,tiff_mpx,percentage_empty
0,0005f7aaab2800f6170c399693a96917,0+0,0,karolinska,True,True,None,None,3,"(27648, 29440)","(1.0, 4.0, 16.0)","(0.45201826153776614, 0.45201826153776614)",96.895192
1,000920ad0b612851f8e01bcc880d9b3d,0+0,0,karolinska,True,True,None,None,3,"(15360, 13312)","(1.0, 4.0, 16.0)","(0.45201826153776614, 0.45201826153776614)",94.424705
2,0018ae58b01bdadc8e347995b69f99aa,4+4,4,radboud,True,True,None,None,3,"(5888, 25344)","(1.0, 4.0, 16.0)","(0.4861876369654638, 0.4861876369654638)",82.860363
3,001c62abd11fa4b57bf7a6c603a11bb9,4+4,4,karolinska,True,True,None,None,3,"(23904, 28664)","(1.0, 4.0, 16.00223338916806)","(0.5031982437947761, 0.5031982437947761)",95.209238
4,001d865e65ef5d2579c190a0e0350d8f,0+0,0,karolinska,True,True,None,None,3,"(28672, 34560)","(1.0, 4.0, 16.0)","(0.45201826153776614, 0.45201826153776614)",94.389209


## Check what came back

Cheap checks, worth doing before trusting anything downstream: did every slide
open, is every file built the same way inside, and how do the two hospitals
compare on size.

In [8]:
"""Confirm the inventory has exactly one row for every slide in train.csv.
This should print True."""

manifest_df.shape[0] == rows

True

In [9]:
"""Count how many slides have a label mask (True) and how many do not (False).

Masks are only ever used to check our work in later stages, never to decide
where on a slide to look.
"""

manifest_df["mask_exists"].value_counts()

mask_exists
True     10516
False      100
Name: count, dtype: int64

In [10]:
"""Check that every slide opened.

A slide that failed to open keeps None in its tiff_pym_lvls column, and
isna() finds those rows. Comparing with == None does not work in pandas: it
returns False for every row, even rows that really are None, so a check
written that way always reports success.

If any slide failed, its id is printed so it can be looked at directly.
"""

failed = manifest_df[manifest_df["tiff_pym_lvls"].isna()]
if len(failed) == 0:
    print("All slides opened correctly")
else:
    print(f"{len(failed)} slides failed to open:")
    print(failed["slide_id"].tolist())

All slides opened correctly


In [11]:
"""Count slides by their number of pyramid levels. If every file is built the
same way, only one value appears here."""

manifest_df["tiff_pym_lvls"].value_counts()

tiff_pym_lvls
3    10616
Name: count, dtype: int64

In [12]:
"""Count slides by their set of downsample factors.

The stored factors are not always exact whole numbers: some slides have
16.0022 rather than 16. Rounding each factor to the nearest whole number groups
those slides with the exact ones, so the count shows the real pattern. round()
is used rather than int(), because int() simply drops the decimals and would
turn a value such as 3.9998 into 3.
"""

downsampling_distribution = manifest_df["tiff_lvl_ds"].apply(lambda x: tuple(round(i) for i in x))
downsampling_distribution.value_counts()

tiff_lvl_ds
(1, 4, 16)    10616
Name: count, dtype: int64

In [13]:
"""Find the smallest slide.

tiff_lvl0_dim holds (width, height) pairs, and taking min() of pairs compares
the widths first, so it would return the narrowest slide rather than the
smallest one. Instead:

1. compute each slide's level-0 area as width times height, skipping any slide
   that failed to open and so has no size;
2. idxmin() gives the row with the smallest area;
3. print that slide's id, hospital and dimensions.
"""

level0_area = manifest_df["tiff_lvl0_dim"].dropna().apply(lambda d: d[0] * d[1])
smallest = manifest_df.loc[level0_area.idxmin()]
print(f"smallest slide: {smallest['slide_id']} ({smallest['provider']}), "
      f"{smallest['tiff_lvl0_dim'][0]} x {smallest['tiff_lvl0_dim'][1]} pixels at level 0")

smallest slide: 0da0915a236f2fc98b299d6fdefe7b8b (radboud), 1280 x 2304 pixels at level 0


## Microns per pixel, per hospital

It is widely repeated that Radboud scans at about 0.24 microns per pixel and
Karolinska at about 0.48 — a factor of two, which would mean a fixed-size tile
covers twice as much tissue on one hospital's slides. The number is written in
these files, so measure it rather than repeating the claim.

In [14]:
"""Microns per pixel, Radboud slides, left to right (x).

1. Take the (x, y) microns-per-pixel pairs for Radboud slides, and separately
   for Karolinska slides.
2. zip(*pairs) splits a list of (x, y) pairs into one sequence of x values and
   one of y values. The names read as: res = resolution, r or k = Radboud or
   Karolinska, x or y = direction.
3. describe() summarises the Radboud x values: count, mean, spread and
   percentiles. A standard deviation of about zero means every Radboud slide
   has the same value.
"""

resolutions_radboud = manifest_df[manifest_df["provider"] == "radboud"]["tiff_mpx"].to_list()
resolutions_karolinska = manifest_df[manifest_df["provider"] == "karolinska"]["tiff_mpx"].tolist()
res_r_x, res_r_y = zip(*resolutions_radboud)
res_k_x, res_k_y = zip(*resolutions_karolinska)
df_r_x = pd.Series(res_r_x)
df_r_x.describe()

count    5.160000e+03
mean     4.861876e-01
std      1.110331e-16
min      4.861876e-01
25%      4.861876e-01
50%      4.861876e-01
75%      4.861876e-01
max      4.861876e-01
dtype: float64

In [15]:
"""Microns per pixel, Karolinska slides, left to right (x). Same summary as
above."""

df_k_x = pd.Series(res_k_x)
df_k_x.describe()

count    5456.000000
mean        0.472590
std         0.025095
min         0.452018
25%         0.452018
50%         0.452018
75%         0.503198
max         0.503198
dtype: float64

In [16]:
"""Microns per pixel, Radboud slides, top to bottom (y)."""

df_r_y = pd.Series(res_r_y)
df_r_y.describe()

count    5.160000e+03
mean     4.861876e-01
std      1.110331e-16
min      4.861876e-01
25%      4.861876e-01
50%      4.861876e-01
75%      4.861876e-01
max      4.861876e-01
dtype: float64

In [17]:
"""Microns per pixel, Karolinska slides, top to bottom (y)."""

df_k_y = pd.Series(res_k_y)
df_k_y.describe()

count    5456.000000
mean        0.472590
std         0.025095
min         0.452018
25%         0.452018
50%         0.452018
75%         0.503198
max         0.503198
dtype: float64

In [18]:
"""Check that pixels are square on every slide.

The four summaries above only show that x and y have the same overall spread.
This checks each slide on its own: count the slides whose x and y values
differ. Zero means every pixel is square, so a single number describes each
slide's resolution. Slides that failed to open have no values and are skipped.
"""

non_square = manifest_df["tiff_mpx"].dropna().apply(lambda xy: xy[0] != xy[1])
print(f"slides where x and y differ: {non_square.sum()} of {len(non_square)}")

slides where x and y differ: 0 of 10616


## Does the grade agree with the Gleason score?

ISUP grade can be derived from the Gleason score, so any row where the two
contradict each other is a labelling error we get for free. Count them, keep
the list, do not edit `train.csv`.

In [19]:
"""The standard mapping from Gleason score to ISUP grade.

A Gleason score is two numbers: the most common tissue pattern plus the second
most common. ISUP groups these into grades 0 to 5:
- 0+0 (no cancer) is grade 0
- 3+3 is grade 1
- 3+4 is grade 2
- 4+3 is grade 3: the same total as 3+4, but worse, because the dominant
  pattern is 4
- 4+4, 3+5 and 5+3 are grade 4
- 4+5, 5+4 and 5+5 are grade 5

The grades are stored as np.int64, the same type as the isup_grade values
they are compared with in the next cell.
"""

gleason_grades = {
    "0+0": np.int64(0),
    "3+3": np.int64(1),
    "3+4": np.int64(2),
    "4+3": np.int64(3),
    "4+4": np.int64(4),
    "5+3": np.int64(4),
    "3+5": np.int64(4),
    "4+5": np.int64(5),
    "5+4": np.int64(5),
    "5+5": np.int64(5),
}

In [20]:
"""Find rows where the ISUP grade contradicts the Gleason score.

1. For each row, look up its Gleason score in the mapping above to get the
   grade it should have.
2. Keep only the rows where the recorded grade is different.

A Gleason score missing from the mapping looks up as None, which never equals a
grade, so such a row is listed here too rather than slipping through. Radboud's
"negative" was already rewritten as "0+0" when the table was built, so it maps
correctly.
"""

manifest_df[manifest_df["isup_grade"] != manifest_df["gleason_score"].apply(gleason_grades.get)]

,slide_id,gleason_score,isup_grade,provider,slide_exists,mask_exists,feature_file_exists,feature_tile_count,tiff_pym_lvls,tiff_lvl0_dim,tiff_lvl_ds,tiff_mpx,percentage_empty
7273,b0a92a74cb53899311acc30b7405e101,4+3,2,karolinska,True,True,None,None,3,"(20916, 35512)","(1.0, 4.0, 16.00333283567217)","(0.5031982437947761, 0.5031982437947761)",94.980748


## Save

Every later stage loads this file instead of reopening ten thousand images.

In [21]:
"""Save the inventory.

1. Create the output folder if it does not exist yet. On a fresh Kaggle
   session it will not.
2. Write the table as parquet. Parquet keeps each column's type, where CSV
   would turn everything back into text. index=False leaves out pandas' row
   numbers, which carry no information here.

Every later stage loads this file instead of reopening all 10,616 images.
"""

os.makedirs(DERIVED_DIR, exist_ok=True)
manifest_df.to_parquet(INVENTORY_PATH, index=False)